# Thunders AI — Robotics

This notebook demonstrates the robotics capabilities of Thunders AI,
including navigation, SLAM mapping, and path planning in simulation environments.

In [ ]:
# Install Thunders AI with robotics support
# !pip install thunders-ai[robotics]

import thunders_ai
from thunders_ai.robotics import Navigation, SLAM, PathPlanner
from thunders_ai.robotics.simulation import SimulationEnv
from thunders_ai.robotics.sensors import LidarSensor, CameraSensor

print(f"Thunders AI version: {thunders_ai.__version__}")

## 1. Set Up Simulation

Create a simulated environment for testing robotics algorithms.

In [ ]:
# Create a simulation environment
env = SimulationEnv(
    world_size=(50, 50),
    resolution=0.1,  # meters per cell
    obstacle_density=0.15,
    seed=42,
)

# Add sensors to the robot
lidar = LidarSensor(
    range_max=10.0,  # meters
    angular_resolution=0.5,  # degrees
    num_beams=720,
)

camera = CameraSensor(
    fov=90,  # degrees
    resolution=(640, 480),
)

env.add_robot(
    position=(5, 5),
    orientation=0.0,
    sensors=[lidar, camera],
)

# Visualize the environment
env.render(title="Simulation Environment")
print(f"Environment: {env.world_size}m, {env.resolution}m resolution")
print(f"Robot position: {env.robot_position}")

## 2. Navigation Example

Use the navigation module for path planning and obstacle avoidance.

In [ ]:
# Initialize the navigation system
nav = Navigation(
    algorithm="a_star",  # Options: a_star, rrt, dijkstra, hybrid_a_star
    safety_margin=0.5,  # meters from obstacles
)

# Set start and goal positions
start = (5, 5)
goal = (45, 45)

# Plan a path
path = nav.plan(
    occupancy_map=env.occupancy_map,
    start=start,
    goal=goal,
)

print(f"Path found: {len(path.waypoints)} waypoints")
print(f"Path length: {path.length:.2f}m")
print(f"Planning time: {path.planning_time_ms:.1f}ms")

# Visualize the planned path
env.render(
    path=path,
    start=start,
    goal=goal,
    title="Planned Path (A*)",
)

In [ ]:
# Simulate the robot following the path with obstacle avoidance
trajectory = nav.follow(
    path=path,
    env=env,
    max_speed=1.0,  # m/s
    obstacle_avoidance=True,
    dynamic_obstacles=True,
)

print(f"Navigation complete: {len(trajectory.positions)} steps")
print(f"Distance traveled: {trajectory.distance:.2f}m")
print(f"Time elapsed: {trajectory.time:.2f}s")
print(f"Collisions: {trajectory.collisions}")

# Animate the trajectory
env.animate(trajectory, title="Robot Navigation")

## 3. SLAM Mapping

Simultaneous Localization and Mapping for building maps from sensor data.

In [ ]:
# Initialize SLAM
slam = SLAM(
    algorithm="gmapping",  # Options: gmapping, cartographer, orb_slam
    resolution=0.1,  # meters per cell
    max_range=10.0,  # sensor max range in meters
)

# Simulate exploration and mapping
num_steps = 200
exploration_path = []

for step in range(num_steps):
    # Get sensor readings
    sensor_data = env.step()
    
    # Update the SLAM map
    slam.update(
        lidar_scan=sensor_data["lidar"],
        odometry=sensor_data["odometry"],
    )
    
    exploration_path.append(env.robot_position)

# Get the generated map
occupancy_map = slam.get_map()
robot_pose = slam.get_pose()

print(f"SLAM map size: {occupancy_map.shape}")
print(f"Estimated pose: x={robot_pose.x:.2f}, y={robot_pose.y:.2f}, θ={robot_pose.theta:.2f}")
print(f"Map coverage: {slam.coverage:.1%}")

In [ ]:
import matplotlib.pyplot as plt

# Visualize the SLAM map
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Ground truth map
axes[0].imshow(env.occupancy_map, cmap="gray_r", origin="lower")
axes[0].set_title("Ground Truth Map")
axes[0].set_xlabel("X (cells)")
axes[0].set_ylabel("Y (cells)")

# SLAM-generated map
axes[1].imshow(occupancy_map, cmap="gray_r", origin="lower")
if exploration_path:
    path_x = [p[0] / env.resolution for p in exploration_path]
    path_y = [p[1] / env.resolution for p in exploration_path]
    axes[1].plot(path_x, path_y, "b-", alpha=0.5, linewidth=1, label="Robot Path")
axes[1].set_title("SLAM-Generated Map")
axes[1].set_xlabel("X (cells)")
axes[1].set_ylabel("Y (cells)")
axes[1].legend()

plt.tight_layout()
plt.savefig("slam_mapping.png", dpi=150, bbox_inches="tight")
plt.show()

print("SLAM map comparison saved to slam_mapping.png")

## 4. Compare Path Planning Algorithms

Compare different path planning algorithms on the same map.

In [ ]:
import time

algorithms = ["a_star", "rrt", "dijkstra", "hybrid_a_star"]
results = []

for algo in algorithms:
    planner = Navigation(algorithm=algo, safety_margin=0.5)
    
    start_time = time.perf_counter()
    path = planner.plan(
        occupancy_map=env.occupancy_map,
        start=start,
        goal=goal,
    )
    elapsed = time.perf_counter() - start_time
    
    results.append({
        "algorithm": algo,
        "path_length": path.length,
        "waypoints": len(path.waypoints),
        "planning_time_ms": elapsed * 1000,
    })

# Display comparison table
print(f"{'Algorithm':<20} {'Path Length':>12} {'Waypoints':>10} {'Time (ms)':>12}")
print("-" * 56)
for r in results:
    print(f"{r['algorithm']:<20} {r['path_length']:>10.2f}m {r['waypoints']:>10} {r['planning_time_ms']:>10.1f}")